# S11 · leakdepth lineage amendment A10 (NB2)

**Purpose.** NB1 found that the frozen lineage matcher (leakdepth 0.1.0, tag `eval-freeze`) reported containment of
0.014–0.448 between OpenFlow corpora that share no rows under exact hashing (S4 negative controls failed in 4/8 directions).
This notebook (1) diagnoses those matches with the frozen code, (2) applies the dated post-result amendment **A10**,
(3) reruns all lineage controls (positive Ahuja vs IoT-DH v1; 8 negative directions) and the S2 lineage injection cells
on held-out seeds 100–119 with **frozen and amended results side by side**, and (4) prints a compact summary.

**Amendment A10 (2026-10-27) — POST-RESULT: specified after seeing the NB1 S4 negative-control results; not part of
frozen design v1.** A candidate row pair is verified only if, in addition to per-column agreement ≥ τ = 0.8, the values
on which it agrees carry at least log2(n_B) + 20 bits of self-information, where each agreeing column contributes
−log2((count + 1)/(n_chunk + 1)) for the partner's value in the streamed chunk of B (canonical hashes, +1 smoothing).
Pairs that pass τ but not this bound are counted as `rejected_low_info`. Because refinement seeds are verified pairs,
they must pass the bound too. Reason: chance agreement on low-information values (binary label, a few emulated host
addresses, small packet/byte counts, zero counters) passed τ. The 20-bit margin was fixed before any amended result.
The frozen wheel is not modified: A10 is a patch that is switched on only inside `amended()`.

**Runtime.** Colab **High-RAM CPU**. Outputs go to `MyDrive/S11-SecSplit/nb2/`; every job is saved as it finishes. After a
disconnect, rerun cells 1–3, then the interrupted cell; finished jobs are skipped. Paste the output of the last cell back.


In [ ]:
# 1. Setup: Drive, embedded frozen leakdepth 0.1.0 wheel (same hash-checked wheel as NB1), versions
import base64, hashlib, json, os, subprocess, sys, time
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/S11-SecSplit/nb2"
os.makedirs(OUT, exist_ok=True)
WHEEL_SHA = "6194495b5dff78f7b98794e04870d535404c1c949d4cfa28a54469abd764737d"  # package files identical to the tested wheel 7da15ab81b61…; author metadata set to the release author
WHEEL_B64 = "..."
w = "/content/leakdepth-0.1.0-py3-none-any.whl"
open(w, "wb").write(base64.b64decode(WHEEL_B64))
assert hashlib.sha256(open(w, "rb").read()).hexdigest() == WHEEL_SHA, "wheel hash mismatch"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", w, "pyarrow", "huggingface_hub"], check=True)
import numpy as np, pandas as pd, sklearn, scipy, leakdepth as L
assert L.__version__ == "0.1.0"
print("leakdepth", L.__version__, "| numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__,
      "| CPUs", os.cpu_count(), "| RAM GB", round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1))
T0 = time.time()

def done(name):
    return os.path.exists(f"{OUT}/{name}")

def jdump(obj, name):
    tmp = f"{OUT}/{name}.tmp"
    json.dump(obj, open(tmp, "w"), default=lambda o: o.item() if hasattr(o, "item") else str(o))
    os.replace(tmp, f"{OUT}/{name}")

def jload(name):
    return json.load(open(f"{OUT}/{name}"))


In [ ]:
# 2. Amendment A10 (POST-RESULT, 2026-10-27) as a switchable patch over the frozen matcher, plus self-tests
import functools, math, importlib
from contextlib import contextmanager
LL = importlib.import_module("leakdepth.lineage")      # the module (leakdepth.lineage is shadowed by the function)
FROZEN_MATCH_ROWS = LL.match_rows
A10_MARGIN_BITS = 20.0
A10_LOG = []

def bits_matrix(H):
    """Self-information (bits) of each value, per column, from its frequency in H (+1 smoothing)."""
    out = np.empty(H.shape, dtype=np.float64); n = H.shape[0]
    for j in range(H.shape[1]):
        _, inv, cnt = np.unique(H[:, j], return_inverse=True, return_counts=True)
        out[:, j] = -np.log2((cnt[inv] + 1.0) / (n + 1.0))
    return out

def agree_mask(ca, ia, cb, ib, rtol=1e-6):
    """Per-column agreement, same rule and column order as the frozen LL.agreement / Canon.col_hashes."""
    a, b = ca.num[ia], cb.num[ib]
    close = np.where(ca.exact, a == b, np.isclose(a, b, rtol=rtol, atol=0.0)) | (np.isnan(a) & np.isnan(b))
    return np.concatenate([close, ca.strh[ia] == cb.strh[ib]], axis=1)

def match_rows_A10(sample, other, a_cols, b_cols, kinds, decimals, tau=0.8, n_bands=8, seed=0, chunk=200_000,
                   margin_bits=None):
    margin = A10_MARGIN_BITS if margin_bits is None else margin_bits
    ca = LL.canonicalise(sample, a_cols, kinds, decimals)
    bands = LL.band_plan(ca.k, n_bands, seed)
    HA = ca.col_hashes()
    BA = np.stack([LL._combine(HA, bd) for bd in bands], axis=1)
    n_other = len(other) if isinstance(other, pd.DataFrame) else None
    n = len(sample)
    matched = np.zeros(n, dtype=bool); partner = np.full(n, -1, dtype=np.int64)
    best = np.zeros(n); info_ok = np.zeros(n); low = np.zeros(n, dtype=bool)
    offset, pairs_for_refine, need = 0, [], float("nan")
    it = LL._chunks(other, chunk) if isinstance(other, pd.DataFrame) else other
    for part in it:
        cb = LL.canonicalise(part, b_cols, kinds, decimals)
        HB = cb.col_hashes()
        bitsB = bits_matrix(HB)
        need = math.log2(max(2, n_other or len(part))) + margin
        for j, bd in enumerate(bands):
            todo = ~matched
            if not todo.any():
                break
            hb = LL._combine(HB, bd)
            first = pd.Series(np.arange(len(hb)), index=hb)
            first = first[~first.index.duplicated()]
            cand = np.flatnonzero(todo & np.isin(BA[:, j], first.index.to_numpy()))
            if len(cand) == 0:
                continue
            ib = first.loc[BA[cand, j]].to_numpy()
            mask = agree_mask(ca, cand, cb, ib)
            ag = mask.sum(axis=1) / max(1, ca.k)
            info = (mask * bitsB[ib]).sum(axis=1)
            best[cand] = np.maximum(best[cand], ag)
            pt = ag >= tau
            ok = pt & (info >= need)
            low[cand[pt & ~ok]] = True
            matched[cand[ok]] = True
            partner[cand[ok]] = ib[ok] + offset
            info_ok[cand[ok]] = info[ok]
            strong = ok & (ag >= 0.999)
            if strong.any() and sum(len(p[0]) for p in pairs_for_refine) < 5000:
                pairs_for_refine.append((cand[strong][:2000], part.iloc[ib[strong][:2000]]))
        offset += len(part)
        if matched.all():
            break
    low &= ~matched
    A10_LOG.append({"n_sample": int(n), "matched": int(matched.sum()), "rejected_low_info": int(low.sum()),
                    "bits_needed": round(need, 2), "k": int(ca.k)})
    return {"matched": matched, "partner": partner, "best_agreement": best, "refine_pairs": pairs_for_refine,
            "info_bits": info_ok, "rejected_low_info": low, "info_needed": need}

@contextmanager
def amended(margin_bits=None):
    LL.match_rows = functools.partial(match_rows_A10, margin_bits=margin_bits)
    try:
        yield
    finally:
        LL.match_rows = FROZEN_MATCH_ROWS

def run_lineage(a, b, na, nb, mode):
    """mode 'frozen' = wheel code unchanged; 'A10' = amended matcher. Returns (report dict, A10 log entries)."""
    del A10_LOG[:]
    if mode == "frozen":
        r = L.lineage(a, b, na, nb, seed=0)
    else:
        with amended():
            r = L.lineage(a, b, na, nb, seed=0)
    d = r.to_dict(); d["a10_log"] = list(A10_LOG)
    return d

# Self-tests on development seeds (0-4) only: (a) A10 with margin -inf reproduces the frozen matcher exactly;
# (b) A10 at the primary margin on a few dev configurations (printed, not a criterion).
from leakdepth.inject import plant
for s, tf in [(0, "rename"), (1, "round3"), (2, "combo"), (3, "renameall"), (4, "colsubset")]:
    p = plant("lineage", 0.05, s, transform=tf)
    f = L.lineage(p.train, p.other, "train", "other", seed=0).to_dict()["results"]["containment"]
    with amended(margin_bits=-1e18):
        g = L.lineage(p.train, p.other, "train", "other", seed=0).to_dict()["results"]["containment"]
    assert [(c["direction"], c["matched"]) for c in f] == [(c["direction"], c["matched"]) for c in g], (s, tf, f, g)
    with amended():
        h = L.lineage(p.train, p.other, "train", "other", seed=0).to_dict()["results"]["containment"]
    print(f"dev seed {s} {tf:9s} planted {p.n_leaked:4d} | frozen", [c["matched"] for c in f], "| A10", [c["matched"] for c in h])
print("self-test passed: A10 with the bound disabled is identical to the frozen matcher")
open(f"{OUT}/A10_amendment.md", "w").write(
    "A10 (2026-10-27), POST-RESULT (after NB1 S4 negative controls). Verified pair additionally needs "
    "sum of self-information of agreeing values >= log2(n_B) + 20 bits (per-column frequencies in the streamed B chunk, "
    "+1 smoothing). Frozen wheel unchanged; patch active only inside amended().\n")


In [ ]:
# 3. Data: S06 raw copies and IoT-DH v1 on Drive; CIC-IDS2017 / UNSW-NB15 HF mirrors (SHA-256 checked, as in NB1)
import glob, zipfile, gc
RAW6 = "/content/drive/MyDrive/S06-OFDG/raw"
RAW11 = "/content/drive/MyDrive/S11-SecSplit/raw"
FILES6 = {"karmous": "karmous__SDN_DDoS_Dataset.csv", "snt": "snt__SNT_dataset.csv", "ukm": "ukmddosflow__flow_features.csv"}
for f in ["ahuja__dataset_sdn.csv", *FILES6.values()]:
    assert os.path.exists(f"{RAW6}/{f}"), f"missing {RAW6}/{f}"

def read6(nm):
    return pd.read_csv(f"{RAW6}/{FILES6[nm]}", low_memory=False)

def iotdh_v1():
    hits = glob.glob(f"{RAW11}/*IoT-DH*.csv") + glob.glob(f"{RAW11}/*IoT-DH*.zip")
    assert hits, f"IoT-DH v1 not found in {RAW11} (NB1 used 'IoT-DH Dataset.zip', v1, 728,853,142 bytes)"
    p = ([h for h in hits if h.endswith(".csv")] or hits)[0]
    if p.endswith(".zip"):
        z = zipfile.ZipFile(p); name = [n for n in z.namelist() if n.endswith(".csv")][0]
        d = pd.read_csv(z.open(name), low_memory=False)
    else:
        d = pd.read_csv(p, low_memory=False)
    assert len(d) == 15_817_209, len(d)
    return d

ahuja = pd.read_csv(f"{RAW6}/ahuja__dataset_sdn.csv", low_memory=False)
assert len(ahuja) == 104_345, len(ahuja)

def hf_mirrors():
    from huggingface_hub import hf_hub_download
    MANIFEST = [{"repo": "rdpahalavan/CIC-IDS2017", "file": "Network-Flows/CICIDS_Flow.parquet", "sha256": "e64f6adfbcb7f96b018cf36c731be64ef4de3353ccd7340b190590734a0dda35"},
                {"repo": "Mouwiya/UNSW-NB15", "file": "data/train-00000-of-00002.parquet", "sha256": "2aada2a26d061111f4e8fb84e716f5f11264fee71abe04697d42cb89e488d047"},
                {"repo": "Mouwiya/UNSW-NB15", "file": "data/train-00001-of-00002.parquet", "sha256": "7c6699ae967567977dee9e9193543b515255f4e1671ca79bf9ae65e7866ffef1"}]
    P = {}
    for m in MANIFEST:
        p = hf_hub_download(m["repo"], m["file"], repo_type="dataset", local_dir=f"/content/data/{m['repo'].replace('/', '__')}")
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == m["sha256"], "mirror changed"
        P[m["file"]] = p
    cic = pd.read_parquet(P["Network-Flows/CICIDS_Flow.parquet"])
    unsw = pd.concat([pd.read_parquet(P[f]) for f in sorted(P) if f.startswith("data/train")], ignore_index=True)
    return cic, unsw
print("Ahuja rows", len(ahuja), "| S06 raw files present")


In [ ]:
# 4. Diagnosis with the FROZEN code: which aligned columns and values make Karmous/SNT rows match Ahuja rows?
def canon_for(used, px, py, x, y):
    a_cols = [u[0] for u in used]; b_cols = [u[1] for u in used]
    kinds = ["num" if px[a].kind == "num" else "str" for a in a_cols]
    decs = []
    for a, b, k in zip(a_cols, b_cols, kinds):
        dd = [d for d in (px[a].decimals, py[b].decimals) if d is not None] if k == "num" else []
        decs.append(min(dd) if dd else None)
    return LL.canonicalise(x, a_cols, kinds, decs), LL.canonicalise(y, b_cols, kinds, decs)

def zero_col(df):
    for c in df.columns:
        n = str(c).lower().replace("_", "")
        if n in ("pktcount", "packetcount"):
            return c
    return None

def diagnose(x, y, nx, ny):
    px, py = LL.profile_columns(x), LL.profile_columns(y)
    al = [t for t in LL.align_columns(px, py) if px[t[0]].kind != "const"]
    sx = (x.sample(n=200_000, random_state=0) if len(x) > 200_000 else x).reset_index(drop=True)
    res, used = LL._containment(sx, y, px, py, al, 0.8, 8, 0, 200_000, min_aligned=4)
    m = res["matched"]; ia = np.flatnonzero(m); ib = res["partner"][m]
    ca, cb = canon_for(used, px, py, sx, y)
    mask = agree_mask(ca, ia, cb, ib)
    bitsB = bits_matrix(cb.col_hashes())
    info = (mask * bitsB[ib]).sum(axis=1); need = math.log2(len(y)) + A10_MARGIN_BITS
    num_cols = [u for u, k in zip(used, ["num" if px[u[0]].kind == "num" else "str" for u in used]) if k == "num"]
    str_cols = [u for u in used if u not in num_cols]
    cols = []
    for j, u in enumerate(num_cols + str_cols):          # same order as the mask
        va = sx.iloc[ia][u[0]].astype(str)
        cols.append({"a": u[0], "b": u[1], "how": u[3], "agree": round(float(mask[:, j].mean()), 3) if len(ia) else None,
                     "distinct_in_matched": int(va.nunique()), "top": va.value_counts().head(3).to_dict(),
                     "median_bits_B": round(float(np.median(bitsB[ib, j])), 1) if len(ia) else None})
    zx, zy = zero_col(sx), zero_col(y)
    out = {"direction": f"{nx} in {ny}", "sampled": len(sx), "matched": int(m.sum()),
           "aligned_first": [(t[0], t[1], t[3]) for t in al], "used_final": [(u[0], u[1], u[3]) for u in used],
           "columns": cols,
           "agreement_q": np.round(np.quantile(res["best_agreement"][m], [0, .5, 1]), 3).tolist() if m.any() else None,
           "info_bits_q": np.round(np.quantile(info, [0, .5, .9, 1]), 1).tolist() if m.any() else None,
           "bits_needed_A10": round(need, 1), "share_matched_below_A10": round(float(np.mean(info < need)), 4) if m.any() else None,
           "zero_pkt_share_matched": round(float((sx.iloc[ia][zx] == 0).mean()), 4) if zx is not None and m.any() else None,
           "zero_pkt_share_all": round(float((sx[zx] == 0).mean()), 4) if zx is not None else None,
           "examples": pd.concat([sx.iloc[ia[:3]][[u[0] for u in used]].add_prefix("A:").reset_index(drop=True),
                                  y.iloc[ib[:3]][[u[1] for u in used]].add_prefix("B:").reset_index(drop=True)], axis=1).astype(str).to_dict("records")}
    return out

for nm in ["snt", "karmous"]:
    if done(f"diag_ahuja_{nm}.json"):
        D = jload(f"diag_ahuja_{nm}.json")
    else:
        other = read6(nm); t0 = time.time()
        D = [diagnose(ahuja, other, "Ahuja", nm), diagnose(other, ahuja, nm, "Ahuja")]
        jdump(D, f"diag_ahuja_{nm}.json"); del other; gc.collect()
        print(f"({time.time() - t0:.0f} s)")
    for d in D:
        print(f"== {d['direction']}: matched {d['matched']}/{d['sampled']}; agreement q {d['agreement_q']}; "
              f"info bits q {d['info_bits_q']} vs A10 need {d['bits_needed_A10']} -> below {d['share_matched_below_A10']}; "
              f"zero-pkt share matched {d['zero_pkt_share_matched']} vs all {d['zero_pkt_share_all']}")
        for c in d["columns"]:
            print(f"   {c['a']:>16} ~ {c['b']:<24} {c['how']:<13} agree {c['agree']}  distinct {c['distinct_in_matched']}  "
                  f"median bits {c['median_bits_B']}  top {list(c['top'])[:3]}")


In [ ]:
# 5. Lineage controls: frozen vs A10 (positive control last; each job saved as it finishes)
NB1_EXPECTED = {"negative_ahuja_karmous": [1496, 89507], "negative_ahuja_snt": [2681, 6993], "negative_ahuja_ukm": [0, 0],
                "negative_cic_unsw": [0, 0], "positive_ahuja_iotdhv1": [104345, 1356]}

def control(name, a, b, na, nb):
    for mode in ("frozen", "A10"):
        fn = f"lineage_{mode}_{name}.json"
        if not done(fn):
            t0 = time.time(); d = run_lineage(a, b, na, nb, mode); d["seconds"] = round(time.time() - t0, 1); jdump(d, fn)
        d = jload(fn)
        cont = [(c["direction"], c["matched"], c["sampled_rows"], round(c["containment"], 5)) for c in d["results"]["containment"]]
        rej = [e["rejected_low_info"] for e in d.get("a10_log", [])]
        print(f"{name:24s} {mode:6s} {cont}  ({d['seconds']} s){'  rejected_low_info per pass ' + str(rej) if mode == 'A10' else ''}")
        if mode == "frozen":
            got = [c[1] for c in cont]
            print("   frozen reproduces NB1:", got == NB1_EXPECTED[name], "" if got == NB1_EXPECTED[name] else f"(NB1 {NB1_EXPECTED[name]})")

for nm in ["ukm", "snt", "karmous"]:
    name = f"negative_ahuja_{nm}"
    if not (done(f"lineage_frozen_{name}.json") and done(f"lineage_A10_{name}.json")):
        other = read6(nm); control(name, ahuja, other, "Ahuja", nm); del other; gc.collect()
    else:
        control(name, None, None, "Ahuja", nm)
if not (done("lineage_frozen_negative_cic_unsw.json") and done("lineage_A10_negative_cic_unsw.json")):
    cic_df, unsw_df = hf_mirrors(); control("negative_cic_unsw", cic_df, unsw_df, "CIC-IDS2017", "UNSW-NB15"); del cic_df, unsw_df; gc.collect()
else:
    control("negative_cic_unsw", None, None, "CIC-IDS2017", "UNSW-NB15")
if not (done("lineage_frozen_positive_ahuja_iotdhv1.json") and done("lineage_A10_positive_ahuja_iotdhv1.json")):
    iot = iotdh_v1(); control("positive_ahuja_iotdhv1", ahuja, iot, "Ahuja", "IoT-DH v1"); del iot; gc.collect()
else:
    control("positive_ahuja_iotdhv1", None, None, "Ahuja", "IoT-DH v1")


In [ ]:
# 6. S2 lineage injection cells, held-out seeds 100-119: frozen vs A10 (resumable JSONL)
import leakdepth.inject as INJ
TRANSFORMS = INJ.TRANSFORMS
EXTRA_TRANSFORMS = getattr(INJ, "EXTRA_TRANSFORMS", ("renameall",))
S2_FILE = f"{OUT}/s2_lineage_heldout.jsonl"
seen = set()
if os.path.exists(S2_FILE):
    seen = {(r["transform"], r["rho"], r["seed"]) for r in map(json.loads, open(S2_FILE))}
todo = [(tf, rho, s) for tf in (*TRANSFORMS, *EXTRA_TRANSFORMS) for rho in (0.0, 0.01, 0.05, 0.20) for s in range(100, 120)
        if (tf, rho, s) not in seen]
print(len(seen), "done,", len(todo), "to run")
t0 = time.time()
for i, (tf, rho, s) in enumerate(todo):
    p = plant("lineage", rho, s, transform=tf)
    row = {"transform": tf, "rho": rho, "seed": s, "n_leaked": int(p.n_leaked), "n_other": len(p.other)}
    for mode in ("frozen", "A10"):
        d = run_lineage(p.train, p.other, "train", "other", mode)
        row[f"{mode}_detected"] = any(f["id"] == "lineage-containment" and f["severity"] in ("WARN", "ERROR") for f in d["findings"])
        row[f"{mode}_matched"] = [c["matched"] for c in d["results"]["containment"]]
        if mode == "A10":
            row["A10_rejected_low_info"] = sum(e["rejected_low_info"] for e in d["a10_log"])
    with open(S2_FILE, "a") as fh:
        fh.write(json.dumps(row) + "\n")
    if (i + 1) % 100 == 0:
        print(i + 1, "configs", f"{time.time() - t0:.0f} s")
S2 = pd.DataFrame(list(map(json.loads, open(S2_FILE))))
tab = S2.groupby(["transform", "rho"]).agg(frozen=("frozen_detected", "sum"), A10=("A10_detected", "sum"), n=("seed", "count")).reset_index()
print(tab.pivot(index="transform", columns="rho", values=["frozen", "A10"]).to_string())


In [ ]:
# 7. Compact summary (paste this output back). Also written to nb2/S11_NB2_summary.json
S2_EXPECTED_FROZEN = {**{tf: {0.0: 0, 0.01: 13, 0.05: 20, 0.2: 20} for tf in ("reorder", "round3", "float32", "colsubset", "rowsubset", "combo")},
                      "rename": {0.0: 0, 0.01: 12, 0.05: 20, 0.2: 20}, "renameall": {0.0: 0, 0.01: 11, 0.05: 20, 0.2: 20},
                      "noise": {0.0: 0, 0.01: 0, 0.05: 0, 0.2: 0}}
summ = {"leakdepth": L.__version__, "A10_margin_bits": A10_MARGIN_BITS, "S4": {}, "S2": {}, "diag": {}, "flags": []}
for f in sorted(glob.glob(f"{OUT}/lineage_frozen_*.json")):
    name = os.path.basename(f)[len("lineage_frozen_"):-5]
    fz, am = jload(f"lineage_frozen_{name}.json"), (jload(f"lineage_A10_{name}.json") if done(f"lineage_A10_{name}.json") else None)
    rows = []
    for i, c in enumerate(fz["results"]["containment"]):
        a = am["results"]["containment"][i] if am else None
        rows.append([c["direction"], c["sampled_rows"], c["matched"], round(c["containment"], 5),
                     a["matched"] if a else None, round(a["containment"], 5) if a else None])
    summ["S4"][name] = {"rows[dir,n,frozen_m,frozen_C,A10_m,A10_C]": rows,
                        "frozen_reproduces_NB1": [r[2] for r in rows] == NB1_EXPECTED.get(name),
                        "A10_rejected_low_info": [e["rejected_low_info"] for e in am.get("a10_log", [])] if am else None,
                        "sec": [fz.get("seconds"), am.get("seconds") if am else None]}
if os.path.exists(S2_FILE):
    S2 = pd.DataFrame(list(map(json.loads, open(S2_FILE))))
    for (tf, rho), g in S2.groupby(["transform", "rho"]):
        summ["S2"].setdefault(tf, {})[str(rho)] = [int(g.frozen_detected.sum()), int(g.A10_detected.sum()), len(g)]
        if int(g.frozen_detected.sum()) != S2_EXPECTED_FROZEN[tf][round(rho, 2)]:
            summ["flags"].append(f"S2 frozen {tf} rho={rho}: {int(g.frozen_detected.sum())} != Stage 5 {S2_EXPECTED_FROZEN[tf][round(rho, 2)]}")
    over = S2[[max(m) > n if n > 0 else False for m, n in zip(S2.A10_matched, S2.n_leaked)]]
    summ["S2_A10_configs_matched_gt_planted"] = len(over)
for f in sorted(glob.glob(f"{OUT}/diag_ahuja_*.json")):
    for d in jload(os.path.basename(f)):
        summ["diag"][d["direction"]] = {"matched": d["matched"], "used": [u[0] + "~" + u[1] for u in d["used_final"]],
                                        "agree": [c["agree"] for c in d["columns"]], "bits_q": d["info_bits_q"],
                                        "need": d["bits_needed_A10"], "below_A10": d["share_matched_below_A10"],
                                        "zero_pkt_matched_vs_all": [d["zero_pkt_share_matched"], d["zero_pkt_share_all"]]}
# Stop rules requested by the author: report, do not adopt, if A10 hurts the positive control or S2 recall
pos = summ["S4"].get("positive_ahuja_iotdhv1", {}).get("rows[dir,n,frozen_m,frozen_C,A10_m,A10_C]", [])
if pos and (pos[0][5] is None or pos[0][5] < 0.99):
    summ["flags"].append("STOP: A10 positive control C(Ahuja in IoT-DH v1) < 0.99")
for tf, d in summ["S2"].items():
    for rho, (fz, am, n) in d.items():
        if tf != "noise" and float(rho) >= 0.05 and am < 0.95 * n:
            summ["flags"].append(f"STOP: A10 S2 recall {tf} rho={rho} = {am}/{n} < 0.95")
        if tf != "noise" and float(rho) == 0.01 and am < fz:
            summ["flags"].append(f"NOTE: A10 lowers S2 recall {tf} rho=0.01: {fz} -> {am} of {n}")
        if float(rho) == 0.0 and am > 0:
            summ["flags"].append(f"A10 false positives {tf} rho=0: {am}/{n}")
for name, s in summ["S4"].items():
    if s["frozen_reproduces_NB1"] is False:
        summ["flags"].append(f"frozen {name} differs from NB1")
summ["runtime_min_this_session"] = round((time.time() - T0) / 60, 1)
jdump(summ, "S11_NB2_summary.json")
print(json.dumps(summ, separators=(",", ":"), default=str))
